# A SevenNet tutorial: fine-tuning
---
This notebook is a simple tutorial for SevenNet\
[paper](https://pubs.acs.org/doi/10.1021/acs.jctc.4c00190)\
[code](https://github.com/MDIL-SNU/SevenNet)\
For fast training, set device into GPU.\
[Runtime] -> [Change runtime type] -> [T4 GPU] -> [Save]

## 0. Installation
First of all, let's install SevenNet to our computer!

In [12]:
import os
working_dir = os.getcwd() # save current path

# This is a latest version of 7net. May not stable!
!git clone -b ase_db https://github.com/MDIL-SNU/SevenNet.git

# install 7net
%cd SevenNet
!pip install -e .

fatal: destination path 'SevenNet' already exists and is not an empty directory.
/content/SevenNet
Obtaining file:///content/SevenNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sevenn (pyproject.toml) ... done
  Created wheel for sevenn: filename=sevenn-0.10.0-0.editable-py3-none-any.whl size=33535 sha256=c99a6e249165341cc0fd8a191513e7fa47473fcf8888f6f7850b3f0dd7f0d42f
  Stored in directory: /tmp/pip-ephem-wheel-cache-i6lriirc/wheels/c7/ca/8c/848f15d66e19fafdb6ec945b9b0b83cf4d16d5541201fc709a
Successfully built sevenn
  Attempting uninstall: sevenn
    Found existing installation: sevenn 0.10.0
    Uninstalling sevenn-0.10.0:
      Successfully uninstalled sevenn-0.10.0


In [6]:
# check if sevenn is installed well
!sevenn -h

usage: sevenn [-h] [-m {train_v1,train_v2}] [-w [WORKING_DIR]] [-l LOG] [-s] [-d] input_yaml

sevenn version=0.10.0, train model based on the input.yaml

positional arguments:
  input_yaml            input.yaml for training

options:
  -h, --help            show this help message and exit
  -m {train_v1,train_v2}, --mode {train_v1,train_v2}
                        main training script to run. Default is train.
  -w [WORKING_DIR], --working_dir [WORKING_DIR]
                        Path to write outputs. Default is cwd.
  -l LOG, --log LOG     Name of logfile. Default is log.sevenn. It never overwrite.
  -s, --screen          Print log to stdout
  -d, --distributed     Set this flag to enable DDP training.


## 1. Download data
Let's train the SevenNet model using 3BPA dataset. The format of dataset is extxyz. The unit of energy is eV and force is eV/Å

In [7]:
!git clone https://github.com/davkovacs/BOTNet-datasets.git
!ls BOTNet-datasets/dataset_3BPA

Cloning into 'BOTNet-datasets'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 57 (delta 13), reused 37 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 28.73 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (13/13), done.
iso_atoms.xyz  test_1200K.xyz  test_600K.xyz  train_300K.xyz
README.md      test_300K.xyz   test_dih.xyz   train_mixedT.xyz


## 2. Training
If you're familiar with CLI (Command-Line Interface), you can simply train the SevenNet! For training, we need a yaml file as an input. Let's make a simple input.yaml file

In [14]:
import os
from tqdm import tqdm
import torch
from torch_geometric.loader import DataLoader
from sevenn.train.trainer import Trainer
import sevenn.train.ase_dataset as ase_dataset

In [16]:
!sevenn_inference --help

usage: sevenn_inference [-h] [-d DEVICE] [-n NTHREADS] [-nw NWORKERS] [-o OUTPUT] [-b BATCH]
                        [-ofg]
                        checkpoint target [target ...]

sevenn version=0.10.0, sevenn_inference. Evaluate sevenn_data/POSCARs/OUTCARs/ase readable using
the model stored in a checkpoint.

positional arguments:
  checkpoint            checkpoint
  target                target files to evaluate.

options:
  -h, --help            show this help message and exit
  -d DEVICE, --device DEVICE
                        cpu/cuda/cuda:x
  -n NTHREADS, --nthreads NTHREADS
                        passed to torch.set_num_threads
  -nw NWORKERS, --nworkers NWORKERS
                        passed to dataloader, if given with -ofg
  -o OUTPUT, --output OUTPUT
                        path to save the outputs
  -b BATCH, --batch BATCH
                        batch size
  -ofg, --on_the_fly_graph_build
                        build graph on the fly to save memory
